# SCENIC+ analysis

For this tutorial, we are processing the medulloblastoma mouse model single cell multiome data from:

[_Shiraishi, R. & Cancila, G. et al. (2024). Cancer-specific epigenome identifies oncogenic hijacking by nuclear factor I family proteins for medulloblastoma progression. Dev. Cell , 59:2302-2319._](https://www.cell.com/developmental-cell/fulltext/S1534-5807(24)00330-7)

This data set contains three samples comprising FACS sorted cells. These cells reflect the progression from healthy precursors to tumor cells:

1. Ptch1GNP: granule neuron precursors, P7
2. PNC: preneoplastic cells, P28
3. Tumor: tumor cells, adult mice

Data was downloaded from the Gene Expression Omnibus:

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE240362

The `GSE240362_RAW.tar` file contains the raw gene expression counts files in HDF5 format (.h5) and ATAC fragment files. This is the typical output you would obtain from the [10X Genomics Cellranger software](https://www.10xgenomics.com/support/software/cell-ranger/latest/getting-started/cr-what-is-cell-ranger).

The ATAC modality (fragment files) were processed with _pycisTopic_ and gene expression (.5) was processed with _Scanpy_. In addition, _cisTarget_ transcription factor binding site motif databases were generated. The SCENIC+ pipelines was then used to infer eRegulon. To see how the processing was done, see the following notebooks:

* [scanpy_rna_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/scanpy_rna_processing.ipynp)
* [pycistopic_atac_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistopic_atac_processing.ipynp)
* [pycistarget_tfmotif_database.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistarget_tfmotif_database.ipynp)
* [scenicplus_pipeline.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/scenicplus_pipeline.ipynp)

For more details on SCENIC+, have a look at:

* [_Bravo González-Blas, C., De Winter, S., Hulselmans, G., Hecker, N., Matetovici, I., Christiaens, V., ... & Aerts, S. (2023). SCENIC+: single-cell multiomic inference of enhancers and gene regulatory networks. Nat. Methods, 20:1355-1367._](https://www.nature.com/articles/s41592-023-01938-4)
* [SCENIC+ documentation](https://scenicplus.readthedocs.io/en/latest/)

In this notebook, we will analyse the output of the SCENIC+ pipeline.

## SCENIC+ object

Let us have a look at the output of the SCENIC+ pipeline:

In [ ]:
import os

scplus_dir = '/home/training/course_dir/data_dir/nhecker/scenicplus_mm10/out'

os.listdir(scplus_dir)

The most important information and results are collated into the SCENIC+ [MuData](https://mudata.readthedocs.io/stable/) object:
* scplusmdata.h5mu: SCENIC+ mudata object

The other ouput files are partially redundant. We will briefly summarize the information that they contain:

Genome and gene annotation:
* `chromsizes.tsv`: the size of each chromsome
* `genome_annotation.tsv`: genome coordinates of genes
* `tf_names.txt`: a list of the gene names of transcription factors (TF)

Gene-enhancer links
* `search_space.tsv`: the region up- and downstream of the transcription start site (TF) of each gene, which was considered to link potential enhancer-regions to the gene

TF motifs that are considered by SCENIC+:
* `ctx_results.hdf5`, `ctx_results.html`: enriched TF binding site motifs in [pycisTopic](https://pycistopic.readthedocs.io/en/latest/features.html) topics or differentially accessible regions (DARs) of clusters of cells compared to the entire genome (consensus peaks) and HTML summary report
* `dem_results.hdf5`, `dem_results.html`: differentially enriched motifs in a or cluster of cells compared to all other topics/clusters and HTML summary report. The DEM analysis can be too strict. For this dataset, we had to relax the adjusted p-value threshold to 0.5. 

TF-target gene links:
* `cistromes_direct.h5ad`, `cistromes_extended.h5ad`: the initial [cistromes](https://en.wikipedia.org/wiki/Cistrome), that is TF-enhancer-gene links, before the gradient boosting machine (GBM) regression step. `direct` indicates that binding site motifs are directly assigned to the TF while `extended` includes motifs that are derived from orthologous TF genes or that where assigned to TF based on the similarity to other motifs

eRegulon related files:
* `tf_to_gene_adj.tsv`: TF-gene links and gene expression correlation
* `region_to_gene_adj.tsv`: region-gene links and correlation between chromatin accessibility of the region and gene expression
* `eRegulon_direct.tsv`, `eRegulons_extended.tsv`: the inferred eRegulon TF-region-gene interactions including metrics
* `AUCell_direct.h5mu`, `AUCell_extended.h5mu`: eRegulon signature enrichment per cell based on AUCell

Chromatin accessibility and gene expression:
* `ACC_GEX.h5mu`: chromatin accessibility and gene expression data stored in a separate MuData object

For most analysis, we only need the SCENIC+ MuData object. We load the SCENIC+ MuData object with the `mudata.read` fuction:

In [ ]:
import mudata

path_scplus = scplus_dir + '/scplusmdata.h5mu'

scplus_mdata = mudata.read(path_scplus)

In [ ]:
scplus_mdata

The MuData object contains the two modalities gene expression `scRNA_counts` and chromatin accessibility `scATAC_counts`. It contains 24,289 cells; data for 23,280 genes and 370,586 accessible regions. 

In addition, there are four modalities for eRegulons with 92 transcription factors (TFs) based on direct TF motif annotations and 161 TFs that include TF motif annotaions based on orthologs or similarity of motifs. These eRegulon modalities contain signature enrichment values per cell inferred with AUCell.

The metadat each cell barcode is specified in the table `obs`. The prefix `scRNA_counts:` indicates that this column originates from the gene expression anndata object while `scATAC_counts:` refers to the pycisTopic object.

In [ ]:
scplus_mdata.obs

The TF-region-gene interactions of the eRegulons are stored in the `direct_e_regulon_metadata` and `extended_e_regulon_metadata` metadata entries. Each row lists a `Region` linked to a `Gene` and a transcription factor (`TF`):

* `Region`: the potential enhancer region
* `Gene`: the target gene
* `importance_R2G`: the importance score for the region-to-gene link
* `rho_R2G`: the correlation between region accessibility and gene expression
* `importance_x_rho`: importance_R2G x rho_R2G
* `TF`: the transcription factor regulation the expression of the Gene
* `importance_TF2G`: the importance score for the TF-to-gene link
* `rho_TF2G`: the correlation between TF and gene expression
* `regulation`: 1=up-regulation, -1=downregulation
* `triplet_rank`: this is a combined rank over the metrics. A lower rank indicates that the
* `is_extended`: `False` means that TF was linked based a direct binding site motif annotation. For `True`, the binding site motif could have been linked to a ortholog of the gene or was inferred by similarity to another motif.

For each eRegulon, we have a general name `eRegulon_name` and a name for the signatures of each modality: gene signature `Gene_signature_name` and region signature `Region_signature_name`.

eRegulons can have four different modes of operation. This is part of their name: `+/+`, `+/-`, `-/+`, `-,-`. We summarize the information from the [SCENIC+ documentation](https://scenicplus.readthedocs.io/en/latest/api.html#gsea-based-approach), here:

| eRegulon mode | TF-to-gene relationship | region-to-gene relationship | biological role |
| --- | --- | --- | --- |
| +/+ | positive (+) | positive (+) | Chromatin is open and TF activates gene expression |
| +/- | positive (+) | negative (-) | When the TF is expressed the target gene is also expressed but the regions linked to the gene are closed. |
| -/+ | negative (-) | positive (+) | When the TF is expressed the target gene is not expressed. When the target gene is expressed, regions linked to this gene are open. TF could be a chromatin closing repressor.|
| -/- | negative (-) | negative (-) | When the TF is expressed the target gene is not expressed. When the target gene is expressed, regions linked to this gene are closed.|

The `+/+` eRegulons indicate transcriptional activators and are easiest to interprete. The other eRegluon modes involve anti-correlation, which can be more difficult to interprete because co-expressed factors may also affect chromatin accessibility. `+/-` and `-/+` can indicate regulation through chromatin closing of silencer regions and enhancer regions, respectively. `-/-` might indicate that the TF acts as trancriptional repressor by binding a silencer region. SCENIC+ uses GBM regression as a core step. Although SCENIC+ improves upon the original SCENIC by integrating enhancer links and chromatin accessibility, you should always be aware of that it merely provides an indication that should be experimentally validated.

In [ ]:
scplus_mdata.uns["direct_e_regulon_metadata"]

The most likely TF-region-gene interactions can simply be selected based on the lowest `triplet_rank`.

In [ ]:
scplus_mdata.uns["direct_e_regulon_metadata"].sort_values('triplet_rank').head(n=20)

You may want to apply additional criteria for selecting interactions for example based on the gene expression, TF expression and correlation.

You can access and visualize the different modalities specifying `scplus_mdata['scRNA_counts']` for gene expression and `scplus_mdata['scATAC_counts']`.

UMAPs can be viewed with the Scanpy plotting command `sc.pl.umap`.

In [ ]:
import scanpy as sc

sc.pl.umap(scplus_mdata['scRNA_counts'], color = ['sample', 'Nfib'], cmap='inferno')

As Scanpy expects the UMAP to be called `umap` and not `UMAP` we have to assign the existing UMAP for the ATAC modality first to `umap`.

In [ ]:
import scanpy as sc

scplus_mdata['scATAC_counts'].obsm['umap'] = scplus_mdata['scATAC_counts'].obsm['UMAP']

sc.pl.umap(scplus_mdata['scATAC_counts'], color = ['sample', 'chr7:13278514-13279014'], cmap='inferno')

## eRegulon specificity

To investigate the activity of eRegulons across our dataset, we combine the gene-based AUCell (short AUC) modalities into a seperate anndata object. 

In [ ]:
import anndata as ann

eRegulon_gene_AUC = ann.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)

We add the metadata from the MuData object.

In [ ]:
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [ ]:
eRegulon_gene_AUC

To visualize our data, we can infer a neighbourhood graph and UMAP directly from the AUC values directly.

In [ ]:
sc.pp.neighbors(eRegulon_gene_AUC, use_rep = "X")

In [ ]:
sc.tl.umap(eRegulon_gene_AUC)

In [ ]:
sc.pl.umap(eRegulon_gene_AUC, color = "scRNA_counts:sample")

### Regulon Specific Score (RSS)

SCENIC+ offers the option to assess the specificity of eRegulons based on the Regulon Specific Score (RSS). This is essentially the Jensen-Shannon divergence between the AUC values of different clusters of cells. Here we compute RSS values for the three different samples instead of clusters.

In [ ]:
from scenicplus.RSS import (regulon_specificity_scores, plot_rss)

rss = regulon_specificity_scores(
    scplus_mudata = scplus_mdata,
    variable = "scRNA_counts:sample",
    modalities = ["direct_gene_based_AUC", "extended_gene_based_AUC"]
)

This creates a table comprising the RSS values of each sample per eRegulon.

In [ ]:
rss_t = rss.transpose()

In [ ]:
rss_t

We can then select the most characteristic eRegulons by sorting the RSS values per sample.

In [ ]:
rss_t.sort_values('Ptch1GNP', inplace=True, ascending=False)
rss_t[0:20]

It is useful to a column for the mode of regulation.

In [ ]:
rss_t['regulation_mode'] = [ ereg.split('_')[2] for ereg in  rss_t.index]
rss_t

This allows us to select activators `+/+`

In [ ]:
rss_t[rss_t['regulation_mode'] == '+/+']

and  typical repressors `-/+`

In [ ]:
rss_t[rss_t['regulation_mode'] == '-/+']

We can plot the eRegulons ordered by their RSS score for each sample using the `plot_rss` function:

In [ ]:
plot_rss(
    data_matrix = rss,
    top_n = 5,
    num_columns = 3
)

eRegluon activity (AUC values) can also be plotted in a UMAP:

In [ ]:
sc.pl.umap(eRegulon_gene_AUC, color = ['Hes6_extended_+/+_(23g)', 'Pbx1_extended_-/+_(13g)'], cmap='inferno')

Summary heatplots of RSS scores are useful for identifying interesting activators and repressors. This can be done with the `heatmap_dotplot` function.

In [ ]:
from scenicplus.plotting.dotplot import heatmap_dotplot

heatmap_dotplot(
    scplus_mudata = scplus_mdata,
    color_modality = "direct_gene_based_AUC",
    size_modality = "direct_region_based_AUC",
    group_variable = "scRNA_counts:sample",
    eRegulon_metadata_key = "direct_e_regulon_metadata",
    color_feature_key = "Gene_signature_name",
    size_feature_key = "Region_signature_name",
    feature_name_key = "eRegulon_name",
    sort_data_by = "direct_gene_based_AUC",
    orientation = "vertical",
    figsize = (5, 25)
)

Some TFs are predicted to be both activators and repressors, for example `Ebf1`. While this can be true for many TFs it is recommended to inspect that their function in other context through literature research and to inspect metrics like anti-correlation of TF and target genes in more details.

## Plotting eRegulons as graphs

To visualise eRegulons as graphs, we can use the `igraph` package. You can use this function to a create graph from an eRegulon table:

In [ ]:
import pandas as pd
import numpy as np
import igraph as ig

def create_eregulon_graph(eregulon_table: pd.DataFrame):
    """
    Generates a iGraph graph based on eRegulon table.

    Parameters
    ----------
    eregulon_table: pd.DataFrame
        eRegulon table should contain 'TF', 'Gene', 'triplet_rank', and 'regulation'columns
        
    Returns
    -------
    ig.Graph
        directed graph with interactions betweem TF target genes, 
        edge weights are given as  -log(triplet_rank/max(triplet_rank)) + defined weight offset
    """
    
    tf_names = list(set(eregulon_name_table['TF']))
    node_names = list(set([str(tf) for tf in tf_names] + [ str(ereg) for ereg in eregulon_name_table['Gene']]))

    # vertex colors
    gene_color = 'gray'
    tf_color = '#87CEEB'

    # edge colors based on mode of regulation
    upreg_color = 'blue'
    downreg_color = 'red'
    
    weight_offset = 0.1

    # add genes as vertices to directed graph, color according to TF or target gene
    graph_eregulon = ig.Graph(directed=True)
    for node in node_names:
        color = gene_color
        if node in tf_names:
            color =  tf_color
        graph_eregulon.add_vertex(node, color=color)

    # add edges based on TF-target gene interaction
    max_rank = np.max(eregulon_name_table['triplet_rank'])

    for tf, gene, rank, regulation in zip(eregulon_name_table['TF'], eregulon_name_table['Gene'], 
                                          eregulon_name_table['triplet_rank'], eregulon_name_table['regulation']
                                         ):    
        color = upreg_color
        if regulation == -1:
            color = downreg_color
        weight = -np.log(rank / max_rank)
        graph_eregulon.add_edge(tf, gene, weight=weight + weight_offset, color=color)

    return graph_eregulon

To avoid an overly crowded graph, it is a good idea to pick the top 25 TF-region-gene interactions based on their `triplet_rank` per eRegulon:

In [ ]:
# Nfia pick top 25 interactions based on triplet score

eregulon_names  = ['Nfia_direct_+/+', 'Nfia_direct_-/+']
eregulon_name_table = scplus_mdata.uns['direct_e_regulon_metadata'][ scplus_mdata.uns['direct_e_regulon_metadata']['eRegulon_name'].isin(eregulon_names) ]
eregulon_name_table = eregulon_name_table.sort_values('triplet_rank')[0:25]
eregulon_name_table

We can use this table as input for `create_eregulon_graph`:

In [ ]:
nfia_graph = create_eregulon_graph(eregulon_name_table)

The graph can then be plotted with `igraph` functions. First, we compute a graph layout. Second, we call the plotting function where we label vertices with gene names and plot the edge size based on the relative `triplet_rank`. Multiple arrows indicate 

In [ ]:
# plot Nfia eRegulon
graph_eregulon = nfia_graph

layout = graph_eregulon.layout(layout='auto')

plotting_parameters = {}
plotting_parameters["layout"] = layout
plotting_parameters["bbox"] = (750, 750)
plotting_parameters["margin"] = 60

ig.plot(graph_eregulon, vertex_label=graph_eregulon.vs['name'], vertex_size=65, edge_width=graph_eregulon.es['weight'],
       **plotting_parameters)

This plot can show up-regulation (blue) and down-regulation (red)

In [ ]:
# Ebf1
eregulon_names  = ['Ebf1_direct_+/+', 'Ebf1_direct_-/+', 'Ebf1_direct_-/-']
eregulon_name_table = scplus_mdata.uns['direct_e_regulon_metadata'][ scplus_mdata.uns['direct_e_regulon_metadata']['eRegulon_name'].isin(eregulon_names) ]
eregulon_name_table = eregulon_name_table.sort_values('triplet_rank')[0:25]
eregulon_name_table

In [ ]:
ebf1_graph = create_eregulon_graph(eregulon_name_table)

In [ ]:
# plot Ebf1 eRegulon
graph_eregulon = ebf1_graph

layout = graph_eregulon.layout(layout='auto')

plotting_parameters = {}
plotting_parameters["layout"] = layout
plotting_parameters["bbox"] = (750, 750)
plotting_parameters["margin"] = 60

ig.plot(graph_eregulon, vertex_label=graph_eregulon.vs['name'], vertex_size=65, edge_width=graph_eregulon.es['weight'],
       **plotting_parameters)